# Raw Data Analysis Workflow

This notebook runs the full raw-data analysis (DLTS-per-pulse, TOF, M/C, FDM,
multi-hit / dead-zone) against a single PyCCAPT `.h5` file. Two file layouts
are accepted automatically:

- **Calibrated bundle** — the output of the data-processing notebook with
  `save_tdc=True` and `save_range=True`: `/df` (calibrated dld) plus
  `/tdc` (raw, linked) and optionally `/range`.
- **Pure raw acquisition** — the file as written by the control software,
  with `/dld` and `/tdc` groups and no calibrated `/df`.

The detector kind (Surface Concept vs RoentDek) is auto-detected from the
linked `/tdc` group. A single dropdown lets you choose whether the species
list comes from the loaded `/range` table or from manually typed peak
windows.

In [ ]:
# Use the interactive ``ipympl`` backend so every figure rendered by the
# raw-data analysis is zoomable / pannable / resizable in the notebook.
# Falls back to the static inline backend on environments where ipympl is
# not installed (the helpers will still draw, just without interactivity).
try:
    get_ipython().run_line_magic('matplotlib', 'widget')   # type: ignore[name-defined]
except Exception:                                         # pragma: no cover
    get_ipython().run_line_magic('matplotlib', 'inline')  # type: ignore[name-defined]
%load_ext autoreload
%autoreload 2

import subprocess
import warnings

import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings("ignore")

from pyccapt.calibration.core import share_variables
from pyccapt.calibration.tutorials.tutorials_helpers import (
    helper_auto_raw_analysis,
    helper_data_loader,
)

variables = share_variables.Variables()

## 1. Pick the `.h5` file

The selected file can be either a calibrated bundle (`/df` + `/tdc` +
optionally `/range`) or a pure raw acquisition file (`/dld` + `/tdc`).

In [ ]:
button = widgets.Button(description='Load dataset')

@button.on_click
def open_file_on_click(_):
    global dataset_path
    folder_path = variables.last_directory
    script = '..//..//data_tools//run_dataset_path_qt.py'
    result = subprocess.run(
        ['python', script, folder_path, 'dataset'],
        capture_output=True,
        text=True,
        shell=False,
    )
    selected_path = result.stdout.strip()
    if selected_path and selected_path != 'No file chosen':
        dataset_path = selected_path
        variables.last_directory = dataset_path
        print(f'Selected: {dataset_path}')

button

## 2. Load the file

The loader auto-detects the file layout: it tries the calibrated `/df`
group first and falls back to the raw `/dld` + `/tdc` groups when `/df`
isn't present. Either way, `variables.data` holds the dld dataframe and
`variables.data_tdc` holds the linked raw timestamps.

When a `/range` group exists in the file (or a sibling `<dataset>_range.h5`
is found next to it), the range table is loaded too — that enables the
*From range file* peak source in the analysis cell below.

In [ ]:
helper_data_loader.load_calibrated_h5(dataset_path, variables)
display(variables.data.head())
if variables.data_tdc is not None:
    display(variables.data_tdc.head())

## 3. Preview signal columns

A quick log-y histogram of every available signal column on
`variables.data` — one or more of `t (ns)`, `t_c (ns)`, `mc (Da)`,
`mc_uc (Da)`. Use this to decide whether the peak windows in the analysis
cell below should be entered in **TOF (ns)** or in **mass/charge (Da)**.

Columns that are absent or all-zero (e.g. `t_c (ns)` on a never-calibrated
file, or `mc (Da)` on a pure raw acquisition file) are reported and skipped.

In [ ]:
helper_auto_raw_analysis.call_signal_preview(variables)

## 4. Run analysis

Three dropdowns control the run:

- **Peak source** — *Manual peak windows* (type up to six rows) or
  *From range file* (use the loaded `/range` table; manual rows are
  disabled in that case).
- **Peak units** — *TOF (ns)* (default) or *Mass/charge (Da)*. The
  user-typed window values are interpreted in the chosen unit, and the
  per-peak ratio table is computed against that signal column. With
  *TOF (ns)* the TOF histogram overlays the species windows; with
  *Mass/charge (Da)* the mc histogram overlays them.
- **Save plots** — *No* (default) or *Yes*. When *Yes*, every figure is
  also saved beside the dataset as SVG + PNG (300 dpi).

Click *Run analysis* to render the full set of plots
(DLTS-per-pulse, TOF, M/C, FDM, multi-hit) plus inline Markdown summaries.

In [ ]:
helper_auto_raw_analysis.call_auto_raw_data_analysis(variables)